In [67]:
import json
import os
from langchain.llms import Ollama
from langchain.text_splitter import CharacterTextSplitter
import openai
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI

In [68]:

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 


llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)


In [69]:
# Initialize OpenAI client (new method)
client = openai.OpenAI(api_key=OPENAI_API_KEY)

def ask_openai(question, model="gpt-4o"):
    """Sends a question to OpenAI's API and returns the response."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# Example Usage
question = "What is differential privacy?"
answer = ask_openai(question)
print("OpenAI Response:", answer)


OpenAI Response: Differential privacy is a concept and framework used in data privacy that aims to provide means to maximize the accuracy of statistical analysis while minimizing the risk of disclosing sensitive information about individuals in the dataset. It is widely regarded as a strong standard for privacy guarantees.

The core idea behind differential privacy is that when an algorithm is applied to two similar datasets that differ by only one individual's data, the output of the algorithm should be nearly the same regardless of whether that individual was included in the dataset or not. This ensures that any single individual's data does not significantly affect the outcome, thereby providing privacy for that individual.

Key elements of differential privacy include:

1. **Randomization**: Introduces random noise into the data or query results to obscure the contribution of any single individual’s data.
2. **Privacy Budget**: Differential privacy operates under a principle called

In [70]:
ollama = Ollama(base_url='http://localhost:11434', model="llama3.1:70b")

In [71]:
# system_prompt = """
# # Hybrid Knowledge Graph and Keyword Extraction Agent

# ## Role
# You are an advanced information extraction agent specialized in building comprehensive knowledge graphs and keyword indexes from unstructured text. Your output is used for semantic search, indexing, reasoning, and data validation.

# ## Objective
# From the provided text, extract:
# 1. **Entities as nodes** with relevant attributes.
# 2. **Relationships between entities** with relevant attributes.
# 3. **High-quality keywords as nodes**, directly linked to the document for fast indexing.
# 4. **The document node must store the content of the document itself.**

# All outputs must be included in a **single, valid JSON object** following the defined structure.

# ## Allowed Labels
# Each keyword node must use one label from the following list:

# { "person", "organization", "location", "event", "date", "work", "law", "product", "language", "scientific_term", "other" }

# If a term doesn't fit any category clearly, use `"other"`.

# ---

# ## Output Format (JSON)
# ```json
# {
#   "nodes": [
#     {
#       "id": "unique_node_id",
#       "label": "nodetype",
#       "attributes": {
#         "key1": "value1"
#       }
#     },
#     {
#       "id": "unique_keyword_id",
#       "label": "keyword_label"
#     },
#     {
#       "id": "docX",
#       "label": "document",
#       "attributes": {
#         "content": "Original document content here."
#       }
#     }
#   ],
#   "relationships": [
#     {
#       "source": "source_node_id",
#       "target": "target_node_id",
#       "type": "RELATIONSHIP_TYPE",
#       "attributes": {
#         "key1": "value1"
#       }
#     },
#     {
#       "source": "unique_keyword_id",
#       "target": "docX",
#       "type": "MENTIONED_IN"
#     }
#   ]
# }

# """


In [72]:

system_prompt = """
You are an information extraction agent that builds a triple-only knowledge graph (KG) from unstructured text for search, indexing, reasoning, and validation.

You MUST return a single valid JSON object. No prose, no comments, no markdown.

----------------------------------------------------------------
OBJECTIVE
----------------------------------------------------------------
From the provided text, extract ONLY triples (subject, predicate, object). All information—entities, keywords, and scalar values (names, dates, numbers, strings)—MUST be represented as nodes connected by predicates.

Exception: Every node MUST include a single node-level property "doc_id" equal to the given DOC_ID (provided in the user message). No other properties are allowed on nodes.

----------------------------------------------------------------
ALLOWED LABELS
----------------------------------------------------------------
Use ONE primary label from:
{ "person", "organization","entertainment", "location", "event", "date", "work", "law", "product", "language", "scientific_term", "keyword", "other" }

For literal/value nodes:
{ "value_text", "value_number", "value_date", "value_boolean" }

If unclear, use "other". Use value_* only for scalar/literal nodes.

----------------------------------------------------------------
COMPOUND ENTITY FUSION (DO NOT SPLIT!)
----------------------------------------------------------------
Do NOT split compound identifiers such as:
- TV/film seasons, volumes, parts, versions, editions, models.
- Examples: "Chicago Fire (season 4)", "Season 4 of Chicago Fire", "iPhone 13 Pro", "Volume II", "Part 1", "v2.0".

Rules:
1) Prefer a single, fused entity node for the specific version/season:
   - "Chicago Fire (season 4)" → one node: id "chicago_fire_season_4", label "entertainment".
2) Do NOT also create separate nodes "chicago_fire" and "season_4" unless the text contains distinct facts that apply exclusively to the base entity apart from that season/version (rare). By default, create ONLY the fused node.
3) If you must create a base entity node (only when independent facts about the base are present), link the fused node with:
   - (fused) -SEASON_OF-> (base) or (fused) -PART_OF-> (base).
   Otherwise, avoid the base node.

----------------------------------------------------------------
ID & NAMING RULES
----------------------------------------------------------------
- Node IDs: lowercase, underscore_separated, unique in the output. No spaces or special chars.
- For compound entities, fuse meaningful tokens:
  • base_title + season/version marker → e.g., "chicago_fire_season_4", "iphone_13_pro".
- Scalar values must be nodes:
  • Dates → value_date with ISO "YYYY-MM-DD" when possible (else YYYY-MM or YYYY).
  • Numbers → value_number with numeric text only (e.g., "23").
  • Text strings → value_text.
- Node display names (if needed) are value_text nodes connected via HAS_NAME; do not store names as properties.

----------------------------------------------------------------
PREDICATES
----------------------------------------------------------------
Use ALL_CAPS with underscores, e.g.:
WORKS_AT, LOCATED_IN, FOUNDED, BORN_ON, HAS_NAME, RELEASE_DATE, PREMIERE_DATE, CONCLUDED_ON, ORDERED_ON, ORDERED_BY, EPISODE_COUNT, EXECUTIVE_PRODUCED_BY, PRODUCED_BY, KEYWORD_OF, SEASON_OF, PART_OF, INSTANCE_OF, TYPE, HAS_CONTENT, etc.

For scalar facts, ALWAYS connect to a value_* node:
- (entity) -HAS_NAME-> (value_text)
- (entity) -PREMIERE_DATE-> (value_date)
- (entity) -CONCLUDED_ON-> (value_date)
- (entity) -EPISODE_COUNT-> (value_number)
- (entity) -ORDERED_ON-> (value_date)
- (entity) -ORDERED_BY-> (organization)

----------------------------------------------------------------
DOC ID REQUIREMENT (IMPORTANT)
----------------------------------------------------------------
- The user message will include a "DOC_ID". You MUST add: "doc_id": DOC_ID to EVERY node object in "nodes".
- No other node properties are allowed.

----------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON)
----------------------------------------------------------------
Return ONLY this JSON structure:

{
  "nodes": [
    { "id": "unique_node_id", "label": "one_label", "doc_id": "DOC_ID" }
  ],
  "relationships": [
    { "source": "node_id", "type": "PREDICATE_NAME", "target": "node_id" }
  ]
}

If nothing can be extracted, return:
{ "nodes": [], "relationships": [] }

----------------------------------------------------------------
EXTRACTION GUIDELINES
----------------------------------------------------------------
1) Entities: persons, organizations, locations, events, works (including seasons), products, laws, languages, scientific terms, keywords, other.
   - Assign the best label. For TV seasons, use "work" for the fused season entity.
2) Property-as-node:
   - No attributes on nodes (except mandatory "doc_id").
   - Convert names/dates/numbers/strings into value_* nodes linked via HAS_* or appropriate predicates.
3) Dates & Numbers:
   - value_date must be ISO if resolvable; otherwise use the coarsest ISO form.
   - value_number must contain only digits (e.g., "23"). For qualitative/approximate values, use value_text.
4) Minimality & Deduplication:
   - Avoid duplicates. Reuse a single fused node for a given compound entity within this output.
5) Compliance:
   - Strict JSON only. No extra fields, no comments.

----------------------------------------------------------------
ILLUSTRATIVE MINI-EXAMPLE (Compound Entity Fused)
----------------------------------------------------------------
Input: "The fourth season of Chicago Fire premiered on October 13, 2015 and concluded on May 17, 2016. The season contained 23 episodes."

Possible output sketch:
{
  "nodes": [
    { "id": "chicago_fire_season_4", "label": "work", "doc_id": "DOC_ID" },
    { "id": "val_2015_10_13", "label": "value_date", "doc_id": "DOC_ID" },
    { "id": "val_2016_05_17", "label": "value_date", "doc_id": "DOC_ID" },
    { "id": "val_23", "label": "value_number", "doc_id": "DOC_ID" }
  ],
  "relationships": [
    { "source": "chicago_fire_season_4", "type": "PREMIERE_DATE", "target": "val_2015_10_13" },
    { "source": "chicago_fire_season_4", "type": "CONCLUDED_ON", "target": "val_2016_05_17" },
    { "source": "chicago_fire_season_4", "type": "EPISODE_COUNT", "target": "val_23" }
  ]
}


"""

In [73]:
# Function to read text from a file
def read_text_from_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


In [74]:
def extract_knowledge_graph(input_text_chunk, doc_id, labels_list):
    prompt = f"{system_prompt}\n\nDocument ID:\n{doc_id}\n\nInput Text:\n{input_text_chunk}"
    response = ask_openai(prompt)
    # Clean up markdown code block markers if present
    if response.strip().startswith("```"):
        response = response.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    
    return response


In [75]:
def process_single_document(doc, combined_graph, labels_list, failed_chunks_file, max_retries=2):
    doc_id = doc["_id"]
    text = doc["text"]
    retries = 0

    while retries < max_retries:
        try:
            response = extract_knowledge_graph(text, doc_id, list(labels_list))
            extracted_graph = json.loads(response)

            for node in extracted_graph.get("nodes", []):
                if "label" in node:
                    labels_list.add(node["label"])
                if node not in combined_graph["nodes"]:
                    combined_graph["nodes"].append(node)

            for rel in extracted_graph.get("relationships", []):
                if rel not in combined_graph["relationships"]:
                    combined_graph["relationships"].append(rel)


            break  # success
        except json.JSONDecodeError as e:
            retries += 1
            print(f"[ERROR] JSONDecodeError on doc {doc_id}, retry {retries}/{max_retries}: {e}")
            if retries >= max_retries:
                with open(failed_chunks_file, "r+", encoding="utf-8") as f:
                    failed_responses = json.load(f)
                    failed_responses.append({
                        "doc_id": doc_id,
                        "text": text,
                        "response": response
                    })
                    f.seek(0)
                    json.dump(failed_responses, f, indent=4)


In [76]:
def process_jsonl_file(input_path, output_path, failed_chunks_file):
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            combined_graph = json.load(f)
    else:
        combined_graph = {"nodes": [], "relationships": []}

    if not os.path.exists(failed_chunks_file):
        with open(failed_chunks_file, "w", encoding="utf-8") as f:
            json.dump([], f, indent=4)

    labels_list = {node["label"] for node in combined_graph["nodes"] if "label" in node}

    with open(input_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            try:
                doc = json.loads(line)
                print(f"\nProcessing document {line_num}: {doc.get('_id')}")
                process_single_document(doc, combined_graph, labels_list, failed_chunks_file)
                with open(output_path, "w", encoding="utf-8") as out_f:
                    json.dump(combined_graph, out_f, indent=4)
            except Exception as e:
                print(f"[ERROR] Failed to process document {line_num}: {e}")

    print(f"\nCompleted. Graph saved at {output_path}.")

In [77]:
# File paths
input_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/datasets/nq/hybrid.jsonl"
output_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/experiment/0_.json"
failed_chunks_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/experiment/failed_0_.json"

# Run the processor
process_jsonl_file(input_file_path, output_file_path, failed_chunks_file_path)


Processing document 1: doc0



Processing document 2: doc1

Processing document 3: doc2

Processing document 4: doc3

Processing document 5: doc4

Processing document 6: doc5

Processing document 7: doc6

Processing document 8: doc7

Processing document 9: doc8

Processing document 10: doc9

Processing document 11: doc10

Processing document 12: doc11

Processing document 13: doc12

Processing document 14: doc13

Processing document 15: doc14

Processing document 16: doc15

Processing document 17: doc16

Processing document 18: doc17

Processing document 19: doc18

Processing document 20: doc19

Processing document 21: doc20

Processing document 22: doc21

Processing document 23: doc22

Processing document 24: doc23

Processing document 25: doc24

Processing document 26: doc25

Processing document 27: doc26

Processing document 28: doc27

Processing document 29: doc28

Processing document 30: doc29

Processing document 31: doc30

Processing document 32: doc31

Processing document 33: doc32

Processing document 34: 

KeyboardInterrupt: 